# PyLLaMa: train on Colab

Fine-tunes **CodeLlama-7b-Instruct** with QLoRA on the Python-docs dataset, publishes the
adapter to Hugging Face, and benchmarks it against GPT-3.5.

1. **Runtime → Change runtime type → GPU** (T4 works; L4/A100 are faster).
2. **Runtime → Run all.**

Everything is saved to Google Drive. Re-running is safe: the dataset isn't rebuilt, training is
skipped if an adapter already exists (set `FORCE_RETRAIN = True` to train again) or resumes from
the last checkpoint, and benchmark answers that were already generated are reused.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Settings

In [ ]:
# --- Edit these -----------------------------------------------------------
MAX_PAGES         = 150     # docs pages to crawl (400 = full library/tutorial/howto/reference)
MAX_TRAIN_SAMPLES = 6000    # cap so 1 epoch fits a free T4 session; None = use everything
EPOCHS            = 1
MAX_LEN           = 768     # tokens per example
BATCH_SIZE        = 2       # T4-safe; raise to 4-8 on A100
GRAD_ACCUM        = 8       # effective batch = BATCH_SIZE * GRAD_ACCUM
FORCE_RETRAIN     = False   # True = train again even if an adapter is already saved
HF_REPO_ID        = "PyLLaMa"        # your HF username is added after login; "" = skip upload
OPENAI_MODEL      = "gpt-3.5-turbo"  # comparison model for the benchmark
# --------------------------------------------------------------------------

DRIVE_DIR = "/content/drive/MyDrive/pyllama"   # survives disconnects

## 2. Mount Drive, get the code, install dependencies

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs(DRIVE_DIR, exist_ok=True)

if os.path.isdir("/content/Pyllama"):
    !git -C /content/Pyllama pull -q
else:
    !git clone -q https://github.com/geraldketu/Pyllama.git /content/Pyllama
%cd /content/Pyllama
!pip install -q -r requirements.txt
# Colab ships an old torchao that makes peft crash when loading adapters; peft doesn't need it.
!pip uninstall -y -q torchao

## 3. Scrape docs.python.org and build the dataset
Pages are cached on Drive, so re-runs skip the network.

In [ ]:
DATA = f"{DRIVE_DIR}/data"
if not os.path.exists(f"{DATA}/processed/train.json"):
    !python -m pyllama.scrape --out {DATA}/raw/python_docs.jsonl --cache-dir {DATA}/cache --max-pages {MAX_PAGES}
    !python -m pyllama.build_dataset --raw {DATA}/raw/python_docs.jsonl --out-dir {DATA}/processed
else:
    print("Dataset already built:", f"{DATA}/processed")

import json
train = json.load(open(f"{DATA}/processed/train.json"))
print(len(train), "training examples")
print(json.dumps(train[0], indent=2)[:800])

## 4. Fine-tune (QLoRA)
Skipped when an adapter is already saved; otherwise resumes from the latest checkpoint on Drive.

In [ ]:
OUT = f"{DRIVE_DIR}/outputs/pyllama-lora"
ADAPTER = f"{OUT}/final"
if os.path.isfile(f"{ADAPTER}/adapter_config.json") and not FORCE_RETRAIN:
    print("Adapter already trained:", ADAPTER, "(set FORCE_RETRAIN = True to train again)")
else:
    cap = f"--max-train-samples {MAX_TRAIN_SAMPLES}" if MAX_TRAIN_SAMPLES else ""
    !python -m pyllama.train --train {DATA}/processed/train.json --val {DATA}/processed/val.json --output-dir {OUT} --load-in-4bit --resume --epochs {EPOCHS} --max-len {MAX_LEN} --batch-size {BATCH_SIZE} --grad-accum {GRAD_ACCUM} {cap}

## 5. Try it

In [ ]:
!python -m pyllama.generate --adapter {ADAPTER} --load-in-4bit "How do I count word frequencies in a list of strings with collections.Counter?"

## 6. Publish to the Hugging Face Hub
Needs a **write** token from https://huggingface.co/settings/tokens. Uploads only the adapter (small), so it takes seconds.

In [ ]:
if HF_REPO_ID:
    from huggingface_hub import notebook_login, whoami
    notebook_login()  # paste a token with WRITE access
    if "/" not in HF_REPO_ID:
        HF_REPO_ID = f"{whoami()['name']}/{HF_REPO_ID}"
    print("Will push to:", HF_REPO_ID)

In [ ]:
# Shown on the model card. Edit if you retrain with different settings.
TRAINING_NOTES = (
    "QLoRA (4-bit NF4) on 5,038 instruction examples built from 150 docs.python.org pages; "
    "1 epoch on an NVIDIA L4 (~40 min). LoRA r=16, alpha=32, dropout 0.05 on q/k/v/o projections; "
    "lr 2e-4 cosine, effective batch 16, max length 768. Final train loss 1.23, validation loss 1.226."
)
if HF_REPO_ID:
    !python -m pyllama.push_to_hub --adapter {ADAPTER} --repo-id {HF_REPO_ID} --training-notes "{TRAINING_NOTES}"
else:
    print("HF_REPO_ID is empty - skipping upload")

## 7. Benchmark vs GPT-3.5
Add your OpenAI key under **🔑 Secrets** (left sidebar) as `OPENAI_API_KEY` and turn on notebook access.
Results are saved to `pyllama/results/` on Drive; PyLLaMa's answers are reused if they already exist.

In [ ]:
from google.colab import userdata

def secret(*names):
    for name in names:
        try:
            value = userdata.get(name)
            if value:
                return value
        except Exception:
            pass
    return None

key = secret("OPENAI_API_KEY", "OPENAI_API")
if key:
    os.environ["OPENAI_API_KEY"] = key  # the OpenAI client only reads this exact name
else:
    print("No OpenAI key found in Colab Secrets - the GPT-3.5 step will be skipped.")

RESULTS = f"{DRIVE_DIR}/results"
os.makedirs(RESULTS, exist_ok=True)
openai_name = OPENAI_MODEL.replace("/", "_")

if not os.path.exists(f"{RESULTS}/pyllama.jsonl"):
    !python -m pyllama.benchmark --results-dir {RESULTS} generate --backend pyllama --adapter {ADAPTER} --load-in-4bit --name pyllama
else:
    print("Reusing PyLLaMa answers:", f"{RESULTS}/pyllama.jsonl")

if key and not os.path.exists(f"{RESULTS}/{openai_name}.jsonl"):
    !python -m pyllama.benchmark --results-dir {RESULTS} generate --backend openai --model {OPENAI_MODEL} --name {openai_name}

compare = [f"{RESULTS}/pyllama.jsonl"] + ([f"{RESULTS}/{openai_name}.jsonl"] if os.path.exists(f"{RESULTS}/{openai_name}.jsonl") else [])
!python -m pyllama.benchmark --results-dir {RESULTS} evaluate {" ".join(compare)}